# Trial Run 3 Scale Monitor

Launch and monitor sharded hybrid YOLO + SAM3 + CellSeg1 inference across many tiles.

In [ ]:
from pathlib import Path
import os, sys, time, subprocess
import pandas as pd
from IPython.display import Image as IPyImage, FileLink, Markdown, clear_output, display

# ---- User configuration ----
REPO = Path.home() / "Desktop/inference_yolo_sam_cellseg"
SCRIPT = REPO / "trial_run_3_hybrid_inference.py"

TILE_DIR = Path.home() / "Desktop/prototype5_whole_image_runs/target_region_6262_13752um/tiles"
BASE_OUT = Path.home() / "Desktop/prototype5_whole_image_runs/target_region_6262_13752um/trial_run_3_scale"

# One GPU: start with N_WORKERS=1. Try 2 only if memory allows.
# Multi-GPU example: GPU_IDS = [0, 1, 2, 3]
N_WORKERS = 1
GPU_IDS = [0]

# Pixel calibration. Set this to the scanner/export calibration for true um units.
MICRONS_PER_PIXEL = float(os.getenv("TRIAL3_MICRONS_PER_PIXEL", "0.25"))

# External assets.
SAM3_REPO = Path.home() / "Desktop/sam31-cgh-training-data/sam3"
SAM31_OUTPUT_ROOT = Path.home() / "Desktop/sam31-cgh-strategy2/outputs/strategy2_41tiles_full_unfreeze_20260625_160623"
SAM31_CHECKPOINT = SAM31_OUTPUT_ROOT / "checkpoints/checkpoint.pt"
YOLO_MODEL_PATH = Path.home() / "Desktop/sam31-cgh-training-data/training_data/reference_models/cellseg1_cgh_p2_yolo_best.pt"
CELLSEG1_REPO = Path.home() / "Desktop/1.Data/training_pa_he_annotation_full/outputs/cellseg1_cluster_live/cellseg1_repo"
CELLSEG1_RUN_DIR = Path.home() / "Desktop/1.Data/training_pa_he_annotation_full/outputs/cellseg1_cluster_live/cellseg1_cgh_p2_41full_20260625_124306"

BASE_OUT.mkdir(parents=True, exist_ok=True)
assert SCRIPT.exists(), SCRIPT
assert TILE_DIR.exists(), TILE_DIR

print("REPO:", REPO)
print("SCRIPT:", SCRIPT)
print("TILE_DIR:", TILE_DIR)
print("BASE_OUT:", BASE_OUT)
print("N_WORKERS:", N_WORKERS)
print("GPU_IDS:", GPU_IDS)
print("MICRONS_PER_PIXEL:", MICRONS_PER_PIXEL)


In [ ]:
# ---- Helpers used by launch, monitor, log tail, and merge cells ----
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def list_tiles():
    return sorted(p for p in TILE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES and not p.name.startswith("._"))

ALL_TILES = list_tiles()
TOTAL_TILES = len(ALL_TILES)

def worker_out_dir(worker_id):
    return BASE_OUT / f"worker_{worker_id:02d}"

def worker_log_path(worker_id):
    return BASE_OUT / f"worker_{worker_id:02d}.log"

def assigned_count(worker_id, n_workers=N_WORKERS):
    return sum(1 for idx, _ in enumerate(ALL_TILES) if idx % n_workers == worker_id)

def count_completed_total(worker_id):
    pred_dir = worker_out_dir(worker_id) / "pred_masks"
    return len(list(pred_dir.glob("*_hybrid_instance_mask.png"))) if pred_dir.exists() else 0

def completed_baseline(worker_id):
    return int(globals().get("COMPLETED_AT_LAUNCH", {}).get(worker_id, 0))

def count_completed_this_launch(worker_id):
    return max(count_completed_total(worker_id) - completed_baseline(worker_id), 0)

# Backward-compatible alias for older ad hoc cells.
def count_completed(worker_id):
    return count_completed_this_launch(worker_id)

def safe_read_csv(path):
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame()
    if list(df.columns) == ["empty"]:
        return pd.DataFrame()
    return df

def worker_csv(worker_id, name):
    return worker_out_dir(worker_id) / name

def worker_csv_rows(worker_id, name):
    return len(safe_read_csv(worker_csv(worker_id, name)))

def read_all_worker_csv(name):
    frames = []
    for worker_id in range(N_WORKERS):
        df = safe_read_csv(worker_csv(worker_id, name))
        if not df.empty:
            df["worker_id"] = worker_id
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def proc_for_worker(worker_id):
    return PROCS.get(worker_id) if "PROCS" in globals() else None

print("Total tiles found:", TOTAL_TILES)
print("Assigned per worker:", {i: assigned_count(i) for i in range(N_WORKERS)})


In [ ]:
# ---- Launch sharded workers ----
# Re-running this cell will not relaunch workers that are still running in this kernel.
PROCS = globals().get("PROCS", {})
active_before_launch = any(proc is not None and proc.poll() is None for proc in PROCS.values())

if not active_before_launch:
    RUN_STARTED_AT = time.time()
    COMPLETED_AT_LAUNCH = {worker_id: count_completed_total(worker_id) for worker_id in range(N_WORKERS)}
    MORPHOLOGY_ROWS_AT_LAUNCH = {
        worker_id: worker_csv_rows(worker_id, "trial_run_3_morphology_features.csv")
        for worker_id in range(N_WORKERS)
    }
    METRICS_ROWS_AT_LAUNCH = {
        worker_id: worker_csv_rows(worker_id, "trial_run_3_metrics.csv")
        for worker_id in range(N_WORKERS)
    }
else:
    RUN_STARTED_AT = globals().get("RUN_STARTED_AT", time.time())
    COMPLETED_AT_LAUNCH = globals().get("COMPLETED_AT_LAUNCH", {})
    MORPHOLOGY_ROWS_AT_LAUNCH = globals().get("MORPHOLOGY_ROWS_AT_LAUNCH", {})
    METRICS_ROWS_AT_LAUNCH = globals().get("METRICS_ROWS_AT_LAUNCH", {})

for worker_id in range(N_WORKERS):
    existing = PROCS.get(worker_id)
    if existing is not None and existing.poll() is None:
        print(f"worker {worker_id:02d} already running pid={existing.pid}")
        continue

    out_dir = worker_out_dir(worker_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = worker_log_path(worker_id)

    COMPLETED_AT_LAUNCH.setdefault(worker_id, count_completed_total(worker_id))
    MORPHOLOGY_ROWS_AT_LAUNCH.setdefault(worker_id, worker_csv_rows(worker_id, "trial_run_3_morphology_features.csv"))
    METRICS_ROWS_AT_LAUNCH.setdefault(worker_id, worker_csv_rows(worker_id, "trial_run_3_metrics.csv"))

    gpu_id = GPU_IDS[worker_id % len(GPU_IDS)]
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu_id),
        "PYTHONUNBUFFERED": "1",
        "PYTHONFAULTHANDLER": "1",
        "TRIAL3_TRACEBACK_AFTER_SECONDS": "180",
        "TRIAL3_IMAGE_DIR": str(TILE_DIR),
        "TRIAL3_TILE_KEYS": "ALL",
        "TRIAL3_SHARD_ID": str(worker_id),
        "TRIAL3_NUM_SHARDS": str(N_WORKERS),
        "TRIAL3_SKIP_EXISTING": "1",
        "TRIAL3_SAVE_COMPARISONS": "1",
        "TRIAL3_COMPARISON_EVERY": "5",
        "TRIAL3_SAVE_DIAGNOSTICS": "0",
        "TRIAL3_VIS_DPI": "120",
        "TRIAL3_CSV_APPEND_EVERY": "5",
        "TRIAL3_OUT_DIR": str(out_dir),
        "SAM3_REPO": str(SAM3_REPO),
        "SAM31_OUTPUT_ROOT": str(SAM31_OUTPUT_ROOT),
        "SAM31_CHECKPOINT": str(SAM31_CHECKPOINT),
        "YOLO_MODEL_PATH": str(YOLO_MODEL_PATH),
        "CELLSEG1_REPO": str(CELLSEG1_REPO),
        "CELLSEG1_RUN_DIR": str(CELLSEG1_RUN_DIR),
    })

    log_handle = log_path.open("a", buffering=1)
    cmd = [sys.executable, str(SCRIPT)]
    proc = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log_handle, stderr=subprocess.STDOUT, text=True)
    PROCS[worker_id] = proc
    print(
        f"launched worker {worker_id:02d} gpu={gpu_id} pid={proc.pid} "
        f"previous_completed={COMPLETED_AT_LAUNCH[worker_id]} out={out_dir} log={log_path}"
    )

print("Launch complete.")
print("Completed baselines:", COMPLETED_AT_LAUNCH)
print("Morphology row baselines:", MORPHOLOGY_ROWS_AT_LAUNCH)


In [ ]:
# ---- Live dashboard. Stop this cell manually when you are done watching. ----
REFRESH_SECONDS = 5

def format_seconds(seconds):
    if seconds is None or pd.isna(seconds) or seconds == float("inf"):
        return "n/a"
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def dashboard_once():
    clear_output(wait=True)
    now = time.time()
    elapsed = now - RUN_STARTED_AT

    completed_at_launch = globals().get("COMPLETED_AT_LAUNCH", {})
    morphology_at_launch = globals().get("MORPHOLOGY_ROWS_AT_LAUNCH", {})
    metrics_at_launch = globals().get("METRICS_ROWS_AT_LAUNCH", {})

    status_rows = []
    total_assigned = 0
    total_previous_completed = 0
    total_completed_this_launch = 0
    total_completed_overall = 0
    for worker_id in range(N_WORKERS):
        proc = proc_for_worker(worker_id)
        running = proc is not None and proc.poll() is None
        return_code = None if proc is None else proc.poll()
        assigned = assigned_count(worker_id)
        completed_total = count_completed_total(worker_id)
        previous_completed = min(int(completed_at_launch.get(worker_id, 0)), completed_total)
        completed_this_launch = max(completed_total - previous_completed, 0)
        total_assigned += assigned
        total_previous_completed += previous_completed
        total_completed_this_launch += completed_this_launch
        total_completed_overall += completed_total
        worker_rate = completed_this_launch / elapsed if elapsed > 0 else 0.0
        remaining = max(assigned - completed_total, 0)
        eta = remaining / worker_rate if worker_rate > 0 else None
        metrics_rows_total = worker_csv_rows(worker_id, "trial_run_3_metrics.csv")
        morph_rows_total = worker_csv_rows(worker_id, "trial_run_3_morphology_features.csv")
        metrics_rows_this_launch = max(metrics_rows_total - int(metrics_at_launch.get(worker_id, 0)), 0)
        morph_rows_this_launch = max(morph_rows_total - int(morphology_at_launch.get(worker_id, 0)), 0)
        status_rows.append({
            "worker_id": worker_id,
            "gpu_id": GPU_IDS[worker_id % len(GPU_IDS)],
            "pid": None if proc is None else proc.pid,
            "status": "running" if running else "finished" if proc is not None else "not launched",
            "return_code": return_code,
            "images_assigned": assigned,
            "previous_completed": previous_completed,
            "images_completed": completed_this_launch,
            "completed_total": completed_total,
            "remaining": remaining,
            "ETA": format_seconds(eta),
            "elapsed": format_seconds(elapsed),
            "metrics_rows": metrics_rows_this_launch,
            "metrics_rows_total": metrics_rows_total,
            "morphology_rows": morph_rows_this_launch,
            "morphology_rows_total": morph_rows_total,
            "log_path": str(worker_log_path(worker_id)),
            "output_dir": str(worker_out_dir(worker_id)),
        })

    tiles_per_sec = total_completed_this_launch / elapsed if elapsed > 0 else 0.0
    remaining_total = max(total_assigned - total_completed_overall, 0)
    eta_total = remaining_total / tiles_per_sec if tiles_per_sec > 0 else None

    metrics_df = read_all_worker_csv("trial_run_3_metrics.csv")
    morph_df = read_all_worker_csv("trial_run_3_morphology_features.csv")
    total_morph_rows = len(morph_df)
    previous_morph_rows = sum(int(v) for v in morphology_at_launch.values())
    morph_rows_this_launch = max(total_morph_rows - previous_morph_rows, 0)
    cells_per_sec = morph_rows_this_launch / elapsed if elapsed > 0 else 0.0
    launch_target = max(total_assigned - total_previous_completed, 0)

    display(Markdown(f"""
## Trial Run 3 Live Dashboard

**Throughput this launch:** `{tiles_per_sec:.3f}` tiles/sec | `{cells_per_sec:.3f}` cells/sec  
**Progress this launch:** `{total_completed_this_launch}/{launch_target}` new tiles  
**Previous completed:** `{total_previous_completed}` tiles | **Total done:** `{total_completed_overall}/{total_assigned}` tiles  
**Estimated remaining:** `{format_seconds(eta_total)}` | **Elapsed:** `{format_seconds(elapsed)}` | **Refresh:** `{REFRESH_SECONDS}s`
"""))

    display(Markdown("### Section 1 — Live worker status"))
    display(pd.DataFrame(status_rows))

    display(Markdown("### Section 2 — Live quantitative metrics"))
    if metrics_df.empty:
        display(Markdown("No metrics CSV rows yet."))
    else:
        metric_cols = ["dice", "iou", "precision", "recall_sensitivity", "specificity"]
        summary = metrics_df.groupby("model", dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
        summary = summary.rename(columns={"recall_sensitivity": "recall"})
        display(summary)

    display(Markdown("### Section 3 — Morphology progress"))
    if morph_df.empty:
        display(Markdown("No morphology CSV rows yet."))
    else:
        total_cells = len(morph_df)
        clear_cells = int((morph_df.get("final_class_name") == "clear_cell_boundary").sum()) if "final_class_name" in morph_df else 0
        compact_cells = int((morph_df.get("final_class_name") == "compact_cell_boundary").sum()) if "final_class_name" in morph_df else 0
        area_scale_um2 = MICRONS_PER_PIXEL ** 2
        morph_numeric = morph_df.copy()
        for col in [
            "cell_area_px",
            "nucleus_area_px",
            "cytoplasm_area_px",
            "nc_ratio",
            "cell_equiv_diameter_px",
            "nucleus_equiv_diameter_px",
        ]:
            if col in morph_numeric:
                morph_numeric[col] = pd.to_numeric(morph_numeric[col], errors="coerce")

        morph_numeric["cell_area_um2"] = morph_numeric.get("cell_area_px", pd.Series(dtype=float)) * area_scale_um2
        morph_numeric["nucleus_area_um2"] = morph_numeric.get("nucleus_area_px", pd.Series(dtype=float)) * area_scale_um2
        morph_numeric["cytoplasm_area_um2"] = morph_numeric.get("cytoplasm_area_px", pd.Series(dtype=float)) * area_scale_um2
        morph_numeric["cell_equiv_diameter_um"] = morph_numeric.get("cell_equiv_diameter_px", pd.Series(dtype=float)) * MICRONS_PER_PIXEL
        morph_numeric["nucleus_equiv_diameter_um"] = morph_numeric.get("nucleus_equiv_diameter_px", pd.Series(dtype=float)) * MICRONS_PER_PIXEL

        morph_summary = pd.DataFrame([{
            "cells_this_launch": morph_rows_this_launch,
            "previous_cells": previous_morph_rows,
            "total_cells_in_output": total_cells,
            "clear_cells_total": clear_cells,
            "compact_cells_total": compact_cells,
            "avg_nc_ratio_all": morph_numeric.get("nc_ratio", pd.Series(dtype=float)).mean(),
            "avg_cell_area_um2_all": morph_numeric["cell_area_um2"].mean(),
            "avg_nucleus_area_um2_all": morph_numeric["nucleus_area_um2"].mean(),
            "avg_cytoplasm_area_um2_all": morph_numeric["cytoplasm_area_um2"].mean(),
        }])
        display(Markdown(f"Calibration: `{MICRONS_PER_PIXEL}` um/px; area columns are in `um^2`."))
        display(morph_summary)

        if "final_class_name" in morph_numeric:
            class_summary = morph_numeric.groupby("final_class_name", dropna=False).agg(
                cells=("final_label", "count"),
                avg_cell_area_um2=("cell_area_um2", "mean"),
                avg_nucleus_area_um2=("nucleus_area_um2", "mean"),
                avg_cytoplasm_area_um2=("cytoplasm_area_um2", "mean"),
                avg_nc_ratio=("nc_ratio", "mean"),
                avg_cell_diameter_um=("cell_equiv_diameter_um", "mean"),
                avg_nucleus_diameter_um=("nucleus_equiv_diameter_um", "mean"),
            ).reset_index()
            display(Markdown("Class-level morphology averages"))
            display(class_summary)

    display(Markdown("### Section 4 — Latest inference images"))
    all_image_paths = []
    for worker_id in range(N_WORKERS):
        all_image_paths.extend((worker_out_dir(worker_id) / "comparison_images").glob("*_trial3_compare.png"))
    new_image_paths = [
        path for path in all_image_paths
        if path.exists() and path.stat().st_mtime >= RUN_STARTED_AT - 10
    ]
    image_paths = sorted(new_image_paths, key=lambda p: p.stat().st_mtime, reverse=True)[:3]
    if not image_paths:
        display(Markdown(
            "No new comparison images from this launch yet. "
            "The fast scale worker saves a comparison image every 5 assigned tiles; "
            "prediction masks still save for every completed tile."
        ))
    for path in image_paths:
        mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime))
        display(FileLink(str(path), result_html_prefix=f"Open ({mtime}): "))
        display(IPyImage(filename=str(path), width=900))

while True:
    dashboard_once()
    time.sleep(REFRESH_SECONDS)


In [ ]:
# ---- Print last 5000 characters of any worker log ----
WORKER_ID = 0
LOG_CHARS = 5000
log_path = worker_log_path(WORKER_ID)
if not log_path.exists():
    print("Log does not exist yet:", log_path)
else:
    text = log_path.read_text(errors="replace")
    print(text[-LOG_CHARS:])


In [ ]:
# ---- Merge worker CSV outputs. Safe to run while workers are still running. ----
MERGE_SPECS = {
    "trial_run_3_metrics.csv": "merged_trial_run_3_metrics.csv",
    "trial_run_3_source_instances.csv": "merged_trial_run_3_source_instances.csv",
    "trial_run_3_final_instances.csv": "merged_trial_run_3_final_instances.csv",
    "trial_run_3_morphology_features.csv": "merged_trial_run_3_morphology_features.csv",
}

for src_name, out_name in MERGE_SPECS.items():
    merged = read_all_worker_csv(src_name)
    out_path = BASE_OUT / out_name
    if merged.empty:
        print(f"No rows yet for {src_name}; skipped {out_path}")
        continue
    merged.to_csv(out_path, index=False)
    print(f"Wrote {len(merged):,} rows -> {out_path}")
